# 07 - sklearn Pipelines: Putting It All Together

**Objective:** Learn how to combine preprocessing steps and models into a single sklearn Pipeline.

**Steps:**
1. Basic Pipeline: StandardScaler → LogisticRegression
2. Pipeline with PCA (dimensionality reduction)
3. Pipeline with RFE (feature selection)
4. Custom KMeans feature transformer
5. GridSearchCV over pipeline parameters
6. Save and load the full pipeline
7. Compare pipeline approaches


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import BaseEstimator, TransformerMixin

import joblib
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully")


Libraries imported successfully


In [2]:
PROCESSED_DIR = Path("../data/processed")
df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")

# Drop rows with missing values
df = df.dropna()

X = df.drop("Diagnosis", axis=1).select_dtypes(include=[np.number])
y = df["Diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts()}")
print(f"Test class distribution:\n{y_test.value_counts()}")


Train shape: (151, 35), Test shape: (38, 35)
Train class distribution:
Diagnosis
0.0    100
1.0     51
Name: count, dtype: int64
Test class distribution:
Diagnosis
0.0    25
1.0    13
Name: count, dtype: int64


### 1. Basic Pipeline: StandardScaler → LogisticRegression

A Pipeline chains multiple transformers and a final estimator. When you call `.fit()`, each step
fits and transforms sequentially (except the last, which only fits). When you call `.predict()`,
each step transforms using the already-fitted parameters — no data leakage.


In [3]:
pipe_basic = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

pipe_basic.fit(X_train, y_train)
y_pred = pipe_basic.predict(X_test)
acc_basic = accuracy_score(y_test, y_pred)

print(f"Basic Pipeline accuracy: {acc_basic:.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['Benign', 'Malignant'])}")


Basic Pipeline accuracy: 0.9474

Classification Report:
              precision    recall  f1-score   support

      Benign       0.93      1.00      0.96        25
   Malignant       1.00      0.85      0.92        13

    accuracy                           0.95        38
   macro avg       0.96      0.92      0.94        38
weighted avg       0.95      0.95      0.95        38



### 2. Pipeline with PCA

Adding PCA inside the Pipeline means scaling and dimensionality reduction are both fitted on
the training set only, then applied consistently to test data.


In [4]:
pipe_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10, random_state=42)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

pipe_pca.fit(X_train, y_train)
y_pred_pca = pipe_pca.predict(X_test)
acc_pca = accuracy_score(y_test, y_pred_pca)

pca = pipe_pca.named_steps["pca"]
print(f"PCA Pipeline accuracy: {acc_pca:.4f}")
print(f"Explained variance with 10 components: {pca.explained_variance_ratio_.sum():.2%}")
print(f"Components: {pca.n_components_}")


PCA Pipeline accuracy: 0.9474
Explained variance with 10 components: 93.99%
Components: 10


### 3. Pipeline with RFE

RFE selects the top K features before passing them to the classifier. By embedding RFE in the
Pipeline, feature selection is re-run on each cross-validation fold — no data leakage.


In [5]:
pipe_rfe = Pipeline([
    ("scaler", StandardScaler()),
    ("rfe", RFE(
        estimator=LogisticRegression(max_iter=1000, random_state=42),
        n_features_to_select=10,
        step=1,
    )),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

pipe_rfe.fit(X_train, y_train)
y_pred_rfe = pipe_rfe.predict(X_test)
acc_rfe = accuracy_score(y_test, y_pred_rfe)

rfe = pipe_rfe.named_steps["rfe"]
selected = X.columns[rfe.support_]
print(f"RFE Pipeline accuracy: {acc_rfe:.4f}")
print(f"Selected {len(selected)} / {X.shape[1]} features:")
print(list(selected))


RFE Pipeline accuracy: 0.9474
Selected 10 / 35 features:
['Mean_Radius', 'Mean_Perimeter', 'SE_Perimeter', 'SE_Area', 'SE_Concavity', 'Worst_Concavity', 'Mean_ConcavePoints', 'SE_ConcavePoints', 'Mean_FractalDimension', 'Texture_Radius_Ratio']


### 4. Custom KMeans Feature Transformer

You can wrap any sklearn-compatible logic into a custom transformer by implementing
`fit()` and `transform()`. Here we use KMeans to add a cluster label as a new feature,
then combine it with the original features via `FeatureUnion`.


In [6]:
class KMeansFeatureAdder(BaseEstimator, TransformerMixin):
    """Fit KMeans and append cluster labels as a new feature column."""
    def __init__(self, n_clusters=3, random_state=42):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)

    def fit(self, X, y=None):
        self.kmeans.fit(X)
        return self

    def transform(self, X):
        clusters = self.kmeans.predict(X).reshape(-1, 1)
        return np.hstack([X, clusters])

pipe_kmeans = Pipeline([
    ("scaler", StandardScaler()),
    ("features", FeatureUnion([
        ("original", "passthrough"),
        ("cluster", KMeansFeatureAdder(n_clusters=3)),
    ])),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

pipe_kmeans.fit(X_train, y_train)
y_pred_kmeans = pipe_kmeans.predict(X_test)
acc_kmeans = accuracy_score(y_test, y_pred_kmeans)

print(f"KMeans + FeatureUnion Pipeline accuracy: {acc_kmeans:.4f}")
print(f"Output feature count: {pipe_kmeans.named_steps['features'].transform(X_train[:1]).shape[1]}")


KMeans + FeatureUnion Pipeline accuracy: 0.9474
Output feature count: 71


### 5. GridSearchCV over Pipeline Parameters

Pipeline parameters are accessed via `step_name__parameter_name` syntax. This lets you tune
preprocessing and model hyperparameters in a single cross-validated search.


In [7]:
param_grid = {
    "pca__n_components": [5, 10, 15, 20],
    "clf__C": [0.01, 0.1, 1.0, 10.0],
}

grid = GridSearchCV(
    pipe_pca, param_grid, cv=5, scoring="accuracy", n_jobs=1, verbose=0,
)
grid.fit(X_train, y_train)

print(f"Best CV accuracy: {grid.best_score_:.4f}")
print(f"Best parameters: {grid.best_params_}")
print(f"\nAll results:")
results = pd.DataFrame(grid.cv_results_)
print(results[["param_pca__n_components", "param_clf__C", "mean_test_score", "rank_test_score"]].sort_values("rank_test_score").to_string(index=False))


Best CV accuracy: 0.9667
Best parameters: {'clf__C': 1.0, 'pca__n_components': 5}

All results:
 param_pca__n_components  param_clf__C  mean_test_score  rank_test_score
                       5          1.00         0.966667                1
                       5         10.00         0.960000                2
                      15          1.00         0.947312                3
                      10          1.00         0.940645                4
                      10         10.00         0.940645                4
                      20          1.00         0.940645                4
                      15          0.10         0.940430                7
                      20          0.10         0.940430                7
                       5          0.10         0.940430                7
                      15         10.00         0.933978               10
                      10          0.10         0.933763               11
                      15    

### 6. Save and Load the Full Pipeline

The entire pipeline — scaler, PCA, and classifier — is saved as a single `joblib` file.
When loaded, it applies all preprocessing automatically before predicting. This eliminates
hardcoded feature names and manual transform steps.


In [8]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save the best pipeline from GridSearchCV
joblib.dump(grid.best_estimator_, MODELS_DIR / "pipeline_model.joblib")
print("Pipeline saved to models/pipeline_model.joblib")

# Load and predict
loaded_pipeline = joblib.load(MODELS_DIR / "pipeline_model.joblib")
y_pred_loaded = loaded_pipeline.predict(X_test)
print(f"Loaded pipeline accuracy: {accuracy_score(y_test, y_pred_loaded):.4f}")
print(f"\nPredictions: {y_pred_loaded[:10]}")


Pipeline saved to models/pipeline_model.joblib
Loaded pipeline accuracy: 0.9211

Predictions: [1. 1. 1. 1. 0. 0. 0. 0. 0. 0.]


### 7. Comparison

All pipelines use the same training/testing data. The table below compares their accuracy:


In [9]:
comparison = pd.DataFrame({
    "Pipeline": [
        "Scaler → LR",
        "Scaler → PCA(10) → LR",
        "Scaler → RFE(10) → LR",
        "Scaler → KMeans+Union → LR",
        "GridSearchCV (best)",
    ],
    "Accuracy": [acc_basic, acc_pca, acc_rfe, acc_kmeans, grid.best_score_],
    "Test Accuracy": [
        accuracy_score(y_test, pipe_basic.predict(X_test)),
        accuracy_score(y_test, pipe_pca.predict(X_test)),
        accuracy_score(y_test, pipe_rfe.predict(X_test)),
        accuracy_score(y_test, pipe_kmeans.predict(X_test)),
        accuracy_score(y_test, grid.best_estimator_.predict(X_test)),
    ],
})

print("Pipeline Comparison Table:")
print(comparison.to_string(index=False))

print("\nAll pipelines achieved similar accuracy. The key benefits of Pipelines are:")
print(" - No data leakage (transforms fit only on training data)")
print(" - Single artifact (preprocessing + model in one joblib file)")
print(" - Consistent transforms between train and inference")
print(" - GridSearchCV tunes preprocessing and model simultaneously")


Pipeline Comparison Table:
                  Pipeline  Accuracy  Test Accuracy
               Scaler → LR  0.947368       0.947368
     Scaler → PCA(10) → LR  0.947368       0.947368
     Scaler → RFE(10) → LR  0.947368       0.947368
Scaler → KMeans+Union → LR  0.947368       0.947368
       GridSearchCV (best)  0.966667       0.921053

All pipelines achieved similar accuracy. The key benefits of Pipelines are:
 - No data leakage (transforms fit only on training data)
 - Single artifact (preprocessing + model in one joblib file)
 - Consistent transforms between train and inference
 - GridSearchCV tunes preprocessing and model simultaneously


### Exercises

1. **Try different classifiers**: Replace `LogisticRegression` with `RandomForestClassifier` or `XGBClassifier` in the pipeline. Compare accuracy.
2. **Add more PCA components**: Change `n_components` in the PCA pipeline to 5, 15, 20, or 30. How does accuracy change?
3. **Pipeline for feature scaling only**: Create a pipeline that only scales features and uses `KNeighborsClassifier`. Compare with unscaled version.
4. **Nested cross-validation**: Use `cross_val_score` on `pipe_basic` with `cv=10` to get a more robust accuracy estimate.
5. **Custom transformer**: Write a custom transformer that drops features with variance below a threshold using `VarianceThreshold` — embed it in a Pipeline.
6. **ColumnTransformer**: If your dataset has mixed types, use `ColumnTransformer` to apply different preprocessing to numeric vs. categorical columns within the same Pipeline.
